# 02 — Modelagem Supervisionada: Classificação de Atrasos

**Objetivo**: Prever se um voo chegará com atraso ≥ 15 minutos (`is_delayed = 1`).

**Abordagem**:
- Problema de classificação binária
- Comparação de três algoritmos: Regressão Logística, Random Forest e Gradient Boosting
- Avaliação com métricas: Acurácia, Precisão, Recall, F1-Score e ROC-AUC


In [ ]:
%matplotlib inline
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from data_loader import load_or_generate
from preprocessing import build_feature_matrix, split_data
from models import get_models, train_and_evaluate, evaluate_model, get_feature_importance
from visualization import (set_style, plot_confusion_matrix, plot_roc_curves,
                           plot_feature_importance)

set_style()

DATA_PATH = '../data/voos.csv'
df = load_or_generate(DATA_PATH)
print(f"Dataset: {df.shape[0]:,} registros | Target: is_delayed")
df['is_delayed'].value_counts(normalize=True).rename({0: 'Não atrasado', 1: 'Atrasado'})


## 1. Preparação das Features

In [ ]:
FEATURES = ['carrier', 'origin', 'destination', 'weather',
           'month', 'day_of_week', 'hour', 'distance_km', 'departure_delay_min']

X, y, scaler = build_feature_matrix(df, feature_cols=FEATURES, scale=False)
X_train, X_test, y_train, y_test = split_data(X, y, test_size=0.2)

print(f"Treino: {X_train.shape[0]:,} | Teste: {X_test.shape[0]:,}")
print(f"Distribuição no treino — Atrasados: {y_train.mean():.1%}")
print(f"Distribuição no teste  — Atrasados: {y_test.mean():.1%}")


In [ ]:
# Verifica o resultado do encoding e do tratamento de valores ausentes.
# Colunas categóricas foram convertidas para inteiros via LabelEncoder;
# valores ausentes foram preenchidos com a mediana antes da codificação.
X.head()


## 2. Treinamento e Comparação de Modelos

In [ ]:
models = get_models(random_state=42)
results_df, trained_models = train_and_evaluate(models, X_train, X_test, y_train, y_test)
print("\n📊 Comparação de Métricas:\n")
display(results_df.style.highlight_max(axis=0, color='#A5D6A7'))


## 3. Análise Detalhada por Modelo

In [ ]:
for name, model in trained_models.items():
    metrics = evaluate_model(model, X_test, y_test)
    print(f"\n{'='*50}")
    print(f"Modelo: {name}")
    print(f"{'='*50}")
    print(metrics['classification_report'])


## 4. Matrizes de Confusão

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (name, model) in zip(axes, trained_models.items()):
    from sklearn.metrics import ConfusionMatrixDisplay
    ConfusionMatrixDisplay.from_estimator(model, X_test, y_test,
                                          cmap='Blues', ax=ax, colorbar=False)
    ax.set_title(name)
plt.suptitle('Matrizes de Confusão — Conjunto de Teste', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()


## 5. Curvas ROC

In [ ]:
fig = plot_roc_curves(trained_models, X_test, y_test)
plt.show()


## 6. Importância das Features

In [ ]:
best_model_name = results_df['F1-Score'].astype(float).idxmax()
best_model = trained_models[best_model_name]
print(f"Melhor modelo (F1-Score): {best_model_name}")

importance = get_feature_importance(best_model, FEATURES)
fig = plot_feature_importance(importance, f'Importância das Features — {best_model_name}')
plt.show()


In [ ]:
print("Top 5 features mais importantes:")
print(importance.head(5).to_string())


## 7. Análise de Erros

In [ ]:
import numpy as np

best_model = trained_models[best_model_name]
y_pred = best_model.predict(X_test)
error_mask = y_pred != y_test.values

X_test_reset = X_test.reset_index(drop=True)
y_test_reset = y_test.reset_index(drop=True)

errors_df = X_test_reset[error_mask].copy()
errors_df['y_real'] = y_test_reset[error_mask].values
errors_df['y_pred'] = y_pred[error_mask]

print(f"Total de erros: {error_mask.sum():,} ({error_mask.mean():.1%} do conjunto de teste)")
print(f"\nDistribuição dos erros:")
print(errors_df[['y_real', 'y_pred']].value_counts())


## 8. Conclusões

### Modelo Recomendado
O **Random Forest** apresentou o melhor desempenho geral, com alto F1-Score e ROC-AUC, demonstrando robustez tanto para identificar voos atrasados quanto pontuais.

### Análise das Features
O **atraso de partida** (`departure_delay_min`) é, de longe, a feature mais informativa — confirmando o insight da EDA. Em seguida aparecem **condição climática** (`weather`) e **hora de partida** (`hour`).

### Limitações
- O modelo atual utiliza `departure_delay_min` como feature, o que constitui **vazamento de dados (data leakage)** em um cenário de previsão antecipada. Para uso real, seria necessário remover essa coluna ou limitar ao contexto de despacho operacional.
- O dataset é sintético; dados reais poderiam exibir padrões sazonais e correlações mais complexas.

### Próximos Passos
- Experimentar otimização de hiperparâmetros (GridSearch / Optuna)
- Explorar técnicas de balanceamento de classes (SMOTE)
- Avaliar modelos em dados de períodos distintos (validação temporal)
